it extreact the required items first then will do the removal keyowrd search
- it first extract the description, androidmanifest path and readme then perform the search for removal keyword


In [2]:
import os
import re
import requests
import pandas as pd
from base64 import b64decode
from dotenv import load_dotenv
from time import sleep

# === Load GitHub tokens ===
load_dotenv("All_Tokens.env")
tokens = [os.getenv(f"GITHUB_TOKEN_{i}") for i in range(1, 7)]
tokens = [t for t in tokens if t]
token_index = 0
if not tokens:
    raise ValueError("❌ No GitHub tokens found in All_Tokens.env")

def get_headers():
    global token_index
    token = tokens[token_index]
    token_index = (token_index + 1) % len(tokens)
    return {"Authorization": f"token {token}"}

# === Define keyword baskets ===
keywords = [
    'example', 'sample', 'demo', 'test', 'debug', 'presentation', 'module', 'components',
    'lib', 'library', 'sdk', 'utils', 'utility', 'plugin', 'widget', 'playground', 'framework',
    'architecture', 'skeleton', 'collection', 'starting point', 'protocol', 'benchmark', 'hackathon',
    'classroom', 'course', 'exercise', 'assignment', 'homework', 'assessment', 'interview', 'asset',
    'template', 'catalog', 'tutorial', 'tool'
]
preceded_by = ['this', 'is a', 'is an', 'our', 'my']
not_preceded_by = ['using', 'with']

def has_removal_context(text, k):
    if re.search(rf'(?<!\S){re.escape(k)}(?!\S)', text, re.IGNORECASE):
        if any(re.search(rf'(?<!\S){re.escape(w)}\s+(\S+\s+){{0,4}}{re.escape(k)}(?!\S)', text, re.IGNORECASE) for w in preceded_by):
            if not any(re.search(rf'{re.escape(n)}\s+(\S+\s+){{0,4}}{re.escape(k)}', text, re.IGNORECASE) for n in not_preceded_by):
                return True
    return False

def fetch_manifest_paths(repo):
    url = f"https://api.github.com/search/code?q=filename:AndroidManifest.xml+repo:{repo}"
    r = requests.get(url, headers=get_headers())
    if r.status_code == 200:
        return [item["path"] for item in r.json().get("items", [])]
    return []

def fetch_readme(repo):
    url = f"https://api.github.com/repos/{repo}/readme"
    r = requests.get(url, headers=get_headers())
    if r.status_code == 200:
        try:
            content = r.json().get("content", "")
            return b64decode(content).decode('utf-8', errors='ignore')
        except:
            return ""
    return ""

# === Paths ===
base_dir = r"C:\Android Mobile App\Step1_URL_Search\Type_1_Searching_Pipeline_July 14"
input_path = os.path.join(base_dir, "step2_manifest_final_output.csv")
output_path = os.path.join(base_dir, "step3_removal_keyword_output.csv")
interim_path = output_path.replace(".csv", "_interim.csv")

# === Load Data ===
if os.path.exists(interim_path):
    print(f"🔄 Resuming from interim: {interim_path}")
    df = pd.read_csv(interim_path)
else:
    df = pd.read_csv(input_path)
    df["removal_keyword_flag"] = "none"
    df["removal_reason"] = "none"
    df["Valid_Repo_Step3"] = "none"

# === Filter unreviewed valid repos ===
to_review = df[(df["Valid_Repo_Step2"] == "yes") & (df["Valid_Repo_Step3"] == "none")]
print(f"🔍 Reviewing {len(to_review)} repos...\n")

# === Main Loop ===
for i, idx in enumerate(to_review.index, 1):
    row = df.loc[idx]
    repo = '/'.join(row["html_url"].strip('/').split('/')[-2:])

    name = row["name"].lower() if pd.notna(row["name"]) else ""
    topics = str(row.get("topics", "")).lower()
    description = str(row.get("description", "")).lower()

    reasons = []

    try:
        manifest_paths = fetch_manifest_paths(repo)
        readme = fetch_readme(repo)

        if any(k in path.lower() for k in keywords for path in manifest_paths):
            reasons.append("manifest_path")

        if any(k in name for k in keywords) or any(k in topics for k in keywords):
            reasons.append("repo_name")

        if any(has_removal_context(description, k) for k in keywords):
            reasons.append("description")

        if any(has_removal_context(readme, k) for k in keywords):
            reasons.append("readme")

        flag = "yes" if reasons else "no"
        df.at[idx, "removal_keyword_flag"] = flag
        df.at[idx, "Valid_Repo_Step3"] = "no" if flag == "yes" else "yes"
        df.at[idx, "removal_reason"] = ", ".join(reasons) if reasons else "none"

    except Exception as e:
        print(f"❌ Error with {repo}: {e}")
        df.at[idx, "removal_keyword_flag"] = "error"
        df.at[idx, "Valid_Repo_Step3"] = "no"
        df.at[idx, "removal_reason"] = "error"
        sleep(1)

    # Save interim progress every 10
    if i % 10 == 0:
        df.to_csv(interim_path, index=False)
        print(f"💾 Interim saved to: {interim_path}")

# === Final Save ===
df.to_csv(output_path, index=False)
print(f"\n✅ Step 3 complete. Output saved to: {output_path}")


🔍 Reviewing 28250 repos...

💾 Interim saved to: C:\Android Mobile App\Step1_URL_Search\Type_1_Searching_Pipeline_July 14\step3_removal_keyword_output_interim.csv
💾 Interim saved to: C:\Android Mobile App\Step1_URL_Search\Type_1_Searching_Pipeline_July 14\step3_removal_keyword_output_interim.csv
💾 Interim saved to: C:\Android Mobile App\Step1_URL_Search\Type_1_Searching_Pipeline_July 14\step3_removal_keyword_output_interim.csv
💾 Interim saved to: C:\Android Mobile App\Step1_URL_Search\Type_1_Searching_Pipeline_July 14\step3_removal_keyword_output_interim.csv
💾 Interim saved to: C:\Android Mobile App\Step1_URL_Search\Type_1_Searching_Pipeline_July 14\step3_removal_keyword_output_interim.csv
💾 Interim saved to: C:\Android Mobile App\Step1_URL_Search\Type_1_Searching_Pipeline_July 14\step3_removal_keyword_output_interim.csv
💾 Interim saved to: C:\Android Mobile App\Step1_URL_Search\Type_1_Searching_Pipeline_July 14\step3_removal_keyword_output_interim.csv
💾 Interim saved to: C:\Android Mob